# 奖励模型 (Reward Model) 训练

> RLHF 第二阶段：用人类偏好数据训练 Bradley-Terry 奖励模型。

## 背景
RLHF 需要 RM 对生成回答打分。RM 通常用 SFT 模型初始化，去掉 lm_head 换成一个标量头。
训练目标：让 chosen 回答得分 > rejected 回答得分。

## 损失函数（Bradley-Terry）
P(chosen > rejected) = sigma(r_chosen - r_rejected)
L = -log sigma(r_c - r_r)

## 架构
- base model (SFT checkpoint, 冻结或微调最后几层)
- value head: Linear(hidden_dim, 1)
- 输入：(prompt, chosen_response, rejected_response) 三元组

## 考察点
- Bradley-Terry 模型推导
- RM 与 PPO/DPO 中 reward 的对应关系
- RM over-optimization 问题


In [ ]:
import torch; import torch.nn as nn
from transformers import LlamaConfig, LlamaModel

class RewardModel(nn.Module):
    def __init__(self, base_model=None, hidden_dim=32):
        super().__init__()
        if base_model:
            self.base = base_model
        else:
            self.base = LlamaModel(LlamaConfig(
                vocab_size=12, hidden_size=hidden_dim, num_hidden_layers=1))
        self.value_head = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids, attention_mask=None):
        # TODO: 取最后一层 hidden state 的最后一个 token
        # 过 value_head 输出标量奖励
        raise NotImplementedError

def bradley_terry_loss(r_chosen, r_rejected):
    # TODO: L = -log sigma(r_c - r_r)
    raise NotImplementedError

# ===== 测试验证 =====
B = 4
rm = RewardModel()
c_ids = torch.randint(0, 12, (B, 10))
try:
    r = rm(c_ids)
    assert r.shape == (B, 1)
    loss = bradley_terry_loss(torch.randn(B,1), torch.randn(B,1))
    assert loss.item() > 0
    print("\u2705 RewardModel 测试通过")
except NotImplementedError:
    print("\u2139 待实现")
